In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Coffe_sales.csv')
print("Shape:", df.shape)
print(df.head())

Shape: (3547, 11)
   hour_of_day cash_type  money    coffee_name Time_of_Day Weekday Month_name  \
0           10      card   38.7          Latte     Morning     Fri        Mar   
1           12      card   38.7  Hot Chocolate   Afternoon     Fri        Mar   
2           12      card   38.7  Hot Chocolate   Afternoon     Fri        Mar   
3           13      card   28.9      Americano   Afternoon     Fri        Mar   
4           13      card   38.7          Latte   Afternoon     Fri        Mar   

   Weekdaysort  Monthsort        Date             Time  
0            5          3  2024-03-01  10:15:50.520000  
1            5          3  2024-03-01  12:19:22.539000  
2            5          3  2024-03-01  12:20:18.089000  
3            5          3  2024-03-01  13:46:33.006000  
4            5          3  2024-03-01  13:48:14.626000  


In [2]:
# Health Check
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)
print()
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
hour_of_day    0
cash_type      0
money          0
coffee_name    0
Time_of_Day    0
Weekday        0
Month_name     0
Weekdaysort    0
Monthsort      0
Date           0
Time           0
dtype: int64

Data types:
hour_of_day      int64
cash_type          str
money          float64
coffee_name        str
Time_of_Day        str
Weekday            str
Month_name         str
Weekdaysort      int64
Monthsort        int64
Date               str
Time               str
dtype: object

Duplicate rows: 0


In [3]:
# Fix Date column from string to datetime
df['Date'] = pd.to_datetime(df['Date'])
print("Date column type after fix:", df['Date'].dtype)

Date column type after fix: datetime64[us]


In [4]:
# Fix Monthsort — original values 1-12 don't distinguish between years
# Updated to YYYYMM format so Jan 2024 and Jan 2025 are treated separately
df['Monthsort'] = df['Date'].dt.year * 100 + df['Date'].dt.month

print(df[['Month_name', 'Monthsort']].drop_duplicates('Monthsort').sort_values('Monthsort'))

     Month_name  Monthsort
0           Mar     202403
175         Apr     202404
343         May     202405
584         Jun     202406
807         Jul     202407
1044        Aug     202408
1316        Sep     202409
1660        Oct     202410
2086        Nov     202411
2345        Dec     202412
2604        Jan     202501
2805        Feb     202502
3228        Mar     202503


In [5]:
# Create MonthStart — first day of each month as a proper date
# Required for Looker Studio time series charts to work correctly
df['MonthStart'] = df['Date'].dt.to_period('M').dt.to_timestamp()

print(df['MonthStart'].unique())

<DatetimeArray>
['2024-03-01 00:00:00', '2024-04-01 00:00:00', '2024-05-01 00:00:00',
 '2024-06-01 00:00:00', '2024-07-01 00:00:00', '2024-08-01 00:00:00',
 '2024-09-01 00:00:00', '2024-10-01 00:00:00', '2024-11-01 00:00:00',
 '2024-12-01 00:00:00', '2025-01-01 00:00:00', '2025-02-01 00:00:00',
 '2025-03-01 00:00:00']
Length: 13, dtype: datetime64[us]


In [ ]:
# Classify each transaction as Weekday or Weekend
# Used for operational analysis and staffing insights
df['day_type'] = np.where(
    df['Weekday'].isin(['Mon', 'Tue', 'Wed', 'Thu', 'Fri']),
    'Weekday',
    'Weekend'
)

print(df['day_type'].value_counts())

In [6]:
# Product cost data — theoretical values created for profit analysis exercise
# These do not reflect actual costs of the real coffee shop
products = {
    'coffee_name': ['Latte', 'Americano', 'Americano with Milk', 'Cappuccino', 'Cortado', 'Espresso', 'Hot Chocolate', 'Cocoa'],
    'cost_to_make': [12.00, 8.00, 10.00, 11.00, 9.00, 6.00, 13.00, 13.00]
}

products_df = pd.DataFrame(products)
merged = df.merge(products_df, on='coffee_name')
profit_summary = merged.groupby('coffee_name').agg(
    total_revenue=('money', 'sum'),
    total_profit=('money', lambda x: (x - merged.loc[x.index, 'cost_to_make']).sum())
).reset_index()

print(profit_summary.sort_values('total_profit', ascending=False))

day_type
Weekday    2658
Weekend     889
Name: count, dtype: int64


In [8]:
# Product cost data — theoretical values created for profit analysis exercise
# These do not reflect actual costs of the real coffee shop
products = {
    'coffee_name': ['Latte', 'Americano', 'Americano with Milk', 'Cappuccino', 'Cortado', 'Espresso', 'Hot Chocolate', 'Cocoa'],
    'cost_to_make': [12.00, 8.00, 10.00, 11.00, 9.00, 6.00, 13.00, 13.00]
}

products_df = pd.DataFrame(products)
merged = df.merge(products_df, on='coffee_name')
profit_summary = merged.groupby('coffee_name').agg(
    total_revenue=('money', 'sum'),
    total_profit=('money', lambda x: (x - merged.loc[x.index, 'cost_to_make']).sum())
).reset_index()

print(profit_summary.sort_values('total_profit', ascending=False))

           coffee_name  total_revenue  total_profit
7                Latte       26875.30      17791.30
1  Americano with Milk       24751.12      16661.12
2           Cappuccino       17439.14      12093.14
0            Americano       14650.26      10138.26
6        Hot Chocolate        9933.46       6345.46
3                Cocoa        8521.16       5414.16
4              Cortado        7384.86       4801.86
5             Espresso        2690.28       1916.28


In [9]:
# Save cleaned dataset
df.to_csv('Coffe_sales_clean.csv', index=False)

# Save profit summary
profit_summary.to_csv('profit_summary.csv', index=False)

print("All files saved successfully.")
print("Final dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

All files saved successfully.
Final dataset shape: (3547, 13)
Columns: ['hour_of_day', 'cash_type', 'money', 'coffee_name', 'Time_of_Day', 'Weekday', 'Month_name', 'Weekdaysort', 'Monthsort', 'Date', 'Time', 'MonthStart', 'day_type']
